In [ ]:
# =========================
# 1. Import Libraries
# =========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# =========================
# 2. Load Dataset
# =========================
data = pd.read_csv("Dataset/weatherHistory.csv")

# =========================
# 3. Data Cleaning
# =========================

# Drop useless columns
data.drop(["Formatted Date", "Summary", "Daily Summary"], axis=1, inplace=True)

# Fix column typo
data.rename(columns={"Loud Cover": "Cloud Cover"}, inplace=True)

# Fill missing values
data.fillna(data.mean(numeric_only=True), inplace=True)

# Remove duplicates
data.drop_duplicates(inplace=True)

# =========================
# 4. Feature Engineering (IMPORTANT)
# =========================

# Rename to match your model
data.rename(columns={
    "Temperature (C)": "T",
    "Pressure (millibars)": "SLP",
    "Humidity": "H",
    "Visibility (km)": "VV",
    "Wind Speed (km/h)": "V"
}, inplace=True)

# Create required features
data["TM"] = data["T"] + 2
data["Tm"] = data["T"] - 2
data["VM"] = data["V"] * 1.5

# Create synthetic PM2.5 (target)
data["PM25"] = (
    data["H"] * 50 +
    data["SLP"] * 0.1 -
    data["VV"] * 5 +
    data["V"] * 2
)

# Keep only required columns
data = data[["T", "TM", "Tm", "SLP", "H", "VV", "V", "VM", "PM25"]]

print("Final Dataset:")
print(data.head())

# =========================
# 5. EDA
# =========================

# Correlation Heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(data.corr(), annot=True)
plt.title("Correlation Heatmap")
plt.show()

# Distribution
plt.figure(figsize=(6, 4))
sns.histplot(data["PM25"], kde=True)
plt.title("PM2.5 Distribution")
plt.show()

# =========================
# 6. Split Data
# =========================
X = data.drop("PM25", axis=1)
y = data["PM25"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 7. Model Training
# =========================
model = DecisionTreeRegressor(
    max_depth=6,
    min_samples_split=5,
    random_state=42
)

model.fit(X_train, y_train)

# =========================
# 8. Prediction
# =========================
y_pred = model.predict(X_test)

# =========================
# 9. Evaluation
# =========================
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\nModel Performance:")
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

# =========================
# 10. Visualization
# =========================
plt.scatter(y_test, y_pred)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.show()

# =========================
# 11. Feature Importance
# =========================
importance = model.feature_importances_
features = X.columns

sns.barplot(x=importance, y=features)
plt.title("Feature Importance")
plt.show()

# =========================
# 12. Save Model
# =========================
joblib.dump(model, "tree_gridcv.pkl")
